# Patch Counts and Percentages

This notebook reports:
- AeroSonic train, test, and env: patch counts for positive and negative classes, plus positive ratios
- Norwegian manifest: patch counts per session using the 1 km rows
- Norwegian manifest: positive percentage per session and radius pair using positives from the target radius and negatives from 10 km

In [10]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [11]:
NORWEGIAN_PATH = Path(r'd:/norwegian_manifest.csv')
AEROSONIC_PATH_TRAIN = Path(r'd:/aerosonic_train_manifest.csv')
AEROSONIC_PATH_TEST = Path(r'd:/aerosonic_test_manifest.csv')
AEROSONIC_PATH_ENV = Path(r'd:/aerosonic_env_manifest.csv')

print('Norwegian manifest exists:', NORWEGIAN_PATH.exists())
print('AeroSonic train manifest exists:', AEROSONIC_PATH_TRAIN.exists())
print('AeroSonic test manifest exists:', AEROSONIC_PATH_TEST.exists())
print('AeroSonic env manifest exists:', AEROSONIC_PATH_ENV.exists())

Norwegian manifest exists: True
AeroSonic train manifest exists: True
AeroSonic test manifest exists: True
AeroSonic env manifest exists: True


In [12]:
# Read only required columns for performance on large files
nor = pd.read_csv(NORWEGIAN_PATH, usecols=['session', 'radius_km', 'num_patches', 'num_positive', 'num_negative'])
aero_train = pd.read_csv(AEROSONIC_PATH_TRAIN, usecols=['fold', 'num_patches', 'num_positive', 'num_negative'])
aero_test = pd.read_csv(AEROSONIC_PATH_TEST, usecols=['num_patches', 'num_positive', 'num_negative'])
aero_env = pd.read_csv(AEROSONIC_PATH_ENV, usecols=['num_patches', 'num_positive', 'num_negative'])

nor['radius_km'] = nor['radius_km'].astype(float)

aero_train['fold'] = pd.to_numeric(aero_train['fold'], errors='coerce')

print(f'Norwegian rows: {len(nor):,}')
print(f'AeroSonic train rows: {len(aero_train):,}')
print(f'AeroSonic test rows: {len(aero_test):,}')
print(f'AeroSonic env rows: {len(aero_env):,}')

Norwegian rows: 150
AeroSonic train rows: 1,479
AeroSonic test rows: 416
AeroSonic env rows: 6


In [13]:
# AeroSonic patch counts by split and class
aerosonic_counts = pd.DataFrame({
    'train': [aero_train['num_positive'].sum(), aero_train['num_negative'].sum(), aero_train['num_patches'].sum()],
    'test': [aero_test['num_positive'].sum(), aero_test['num_negative'].sum(), aero_test['num_patches'].sum()],
    'env': [aero_env['num_positive'].sum(), aero_env['num_negative'].sum(), aero_env['num_patches'].sum()],
}, index=['positive_patches', 'negative_patches', 'total_patches'])
aerosonic_counts.index.name = 'class'

aerosonic_ratios = pd.DataFrame({
    'train': [100.0 * aero_train['num_positive'].sum() / aero_train['num_patches'].sum() if aero_train['num_patches'].sum() else 0.0],
    'test': [100.0 * aero_test['num_positive'].sum() / aero_test['num_patches'].sum() if aero_test['num_patches'].sum() else 0.0],
    'env': [100.0 * aero_env['num_positive'].sum() / aero_env['num_patches'].sum() if aero_env['num_patches'].sum() else 0.0],
}, index=['positive_ratio_pct'])

In [14]:
# AeroSonic train positive percentage by fold and overall
train_fold_positive = (
    aero_train.groupby('fold', as_index=False)[['num_positive', 'num_patches', 'num_negative']]
      .sum()
)
train_fold_positive['positive_pct'] = (
    100.0 * train_fold_positive['num_positive'] / train_fold_positive['num_patches']
)
train_fold_positive = train_fold_positive.rename(columns={
    'num_positive': 'positive_patches',
    'num_negative': 'negative_patches',
    'num_patches': 'total_patches',
}).sort_values('fold').reset_index(drop=True)

train_total_positive = pd.DataFrame([
    {
        'dataset': 'aerosonic_train',
        'positive_patches': aero_train['num_positive'].sum(),
        'negative_patches': aero_train['num_negative'].sum(),
        'total_patches': aero_train['num_patches'].sum(),
        'positive_pct': 100.0 * aero_train['num_positive'].sum() / aero_train['num_patches'].sum()
        if aero_train['num_patches'].sum() else 0.0,
    }
])

```latex
\begin{table}[htbp]
    \centering
    \caption{Class distribution (\%) across the five training folds, the aggregated training set, the test set, and the environmental recordings (\textit{Env}) in the AeroSonicDB-YPAD0523 dataset after preprocessing to patches.}
    \label{tab:class-distributions}
    \begin{tabular}{lcccccccc}
        \toprule
        \textbf{Class} & \textbf{Fold 1} & \textbf{Fold 2} & \textbf{Fold 3} & \textbf{Fold 4} & \textbf{Fold 5} & \textbf{Train} & \textbf{Test} & \textbf{Env} \\
        \midrule
        No aircraft & 31.3821\% & 22.6466\% & 37.3544\% & 34.7009\% & 35.3592\% & 32.6061\% & 39.6154\% & 80.0542\% \\
        Aircraft    & 68.6179\% & 77.3534\% & 62.6456\% & 65.2991\% & 64.6408\% & 67.3939\% & 60.3846\% & 19.9458\% \\
        Positive Ratio & 68.6179\% & 77.3534\% & 62.6456\% & 65.2991\% & 64.6408\% & 67.3939\% & 60.3846\% & 19.9458\% \\
        \bottomrule
    \end{tabular}
\end{table}
```

In [15]:
# Norwegian patch counts per session using the 1 km rows
nor = nor.copy()
nor['session'] = nor['session'].replace({'260326_part1': '260326', '260326_part2': '260326'})

nor_1km = nor[nor['radius_km'] == 1.0].copy()
session_patch_counts = (
    nor_1km.groupby('session', as_index=False)['num_patches']
          .sum()
          .rename(columns={'num_patches': 'patches_at_1km'})
          .sort_values('session')
)

total_patches_all_sessions = session_patch_counts['patches_at_1km'].sum()
session_patch_counts['session_share_pct'] = (
    100.0 * session_patch_counts['patches_at_1km'] / total_patches_all_sessions
).round(4)

```latex
\begin{table}[ht]
\centering
\caption{Number of audio patches extracted from each Stjørdal recording session after preprocessing, and each session's share of all patches.}
\label{tab:stjordal-session-patches}
\begin{tabular}{lcc}
\toprule
\textbf{Session} & \textbf{Number of Patches} & \textbf{Share of All Patches} \\
\midrule
030326 & 33622 & 11.2220\% \\
230226 & 45333 & 15.1307\% \\
260326 & 156582 & 52.2621\% \\
280126 & 49446 & 16.5035\% \\
300925 & 14626 & 4.8817\% \\
\bottomrule
\end{tabular}
\end{table}
```

In [16]:
# Norwegian positive percentage for each session and radius pair
nor_positive = (
    nor.groupby(['session', 'radius_km'], as_index=False)['num_positive']
       .sum()
       .rename(columns={'num_positive': 'positive_patches'})
)

nor_negative_10km = (
    nor.loc[nor['radius_km'] == 10.0, ['session', 'num_negative']]
      .groupby('session', as_index=False)['num_negative']
      .sum()
      .rename(columns={'num_negative': 'negative_patches_10km'})
)

nor_positive_pct = nor_positive.merge(nor_negative_10km, on='session', how='left')
nor_positive_pct['positive_pct'] = (
    100.0 * nor_positive_pct['positive_patches']
    / (nor_positive_pct['positive_patches'] + nor_positive_pct['negative_patches_10km'])
)
nor_positive_pct = nor_positive_pct[nor_positive_pct['radius_km'] <= 5.0].sort_values(['session', 'radius_km']).reset_index(drop=True)

```latex
\begin{table}[ht]
\centering
\caption{Percentage of positive (\textit{Aircraft}) patches in each Stjørdal session for different geofence radii.}
\label{tab:positive-percentage-stjordal}
\begin{tabular}{lccccc}
\toprule
\textbf{Session} & \textbf{1 km} & \textbf{2 km} & \textbf{3 km} & \textbf{4 km} & \textbf{5 km} \\
\midrule
030326 & 0.0000\% & 0.0000\% & 0.1116\% & 5.0614\% & 38.5213\% \\
230226 & 0.0000\% & 4.9571\% & 9.4647\% & 22.0338\% & 55.5733\% \\
260326 & 0.0842\% & 1.8202\% & 6.4085\% & 12.0944\% & 17.8514\% \\
280126 & 0.2441\% & 2.5768\% & 5.9676\% & 13.2718\% & 30.9547\% \\
300925 & 37.8013\% & 55.5929\% & 74.8992\% & 80.6917\% & 82.4092\% \\
\bottomrule
\end{tabular}
\end{table}
```

In [17]:
# Optional: save tables
base_dir = Path.cwd()
if base_dir.name == 'notebooks':
    base_dir = base_dir.parent
out_dir = base_dir / 'outputs' / 'exploration'
out_dir.mkdir(parents=True, exist_ok=True)

aerosonic_counts.to_csv(out_dir / 'aerosonic_patch_counts_by_split.csv')
aerosonic_ratios.to_csv(out_dir / 'aerosonic_positive_ratio_by_split.csv')
train_fold_positive.to_csv(out_dir / 'aerosonic_train_positive_pct_by_fold.csv', index=False)
train_total_positive.to_csv(out_dir / 'aerosonic_train_total_positive_pct.csv', index=False)
session_patch_counts.to_csv(out_dir / 'norwegian_session_patch_counts_1km.csv', index=False)
nor_positive_pct.to_csv(out_dir / 'norwegian_session_radius_positive_pct.csv', index=False)

print('Saved CSV summaries to:', out_dir.resolve())

Saved CSV summaries to: C:\Users\imborhau\Documents\sound-event-detection-aircrafts\outputs\exploration
